# 04 · Graph Embeddings

Train Node2Vec on the knowledge graph and use the resulting embeddings for similarity and link prediction.

**Why Node2Vec, not a GNN?** For a graph this small (~50 nodes, ~60 edges), GNNs overfit immediately. Node2Vec is the appropriate tool. If we grow the graph to thousands of nodes (state-level detail, finer industry decomposition), revisit with GraphSAGE.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve() / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from migration_atlas.models.graph_embeddings import (
    EmbeddingsConfig, train_embeddings, load_embeddings, most_similar, link_score,
)

## 1. Train

In [ ]:
cfg = EmbeddingsConfig(
    dimensions=64,
    walk_length=16,
    num_walks=100,
    output_path='../checkpoints/node2vec.kv',
)
train_embeddings(cfg)

## 2. Similarity queries

In [ ]:
kv = load_embeddings('../checkpoints/node2vec.kv')

for query_node in ['mexico', 'india', 'cuba', 'h-1b', 'ina-1965']:
    print(f'\n=== Most similar to: {query_node!r} ===')
    for n, s in most_similar(query_node, top_k=8, kv=kv):
        print(f'  {s:.3f}  {n}')

Reading the results:
- Countries that share visa pathways and industries should cluster — Mexico, El Salvador, Honduras, Guatemala (Central American corridor).
- Visas created by the same law should cluster — H-1B, EB-5, DV Lottery (all 1990 Act).
- The 1965 INA should be most similar to the countries it enabled (India, China, Philippines, etc.).

## 3. Link prediction

Score the plausibility of edges that aren't currently in the graph. High scores might indicate gaps in our seed data.

In [ ]:
candidate_edges = [
    ('brazil', 'tech'),       # plausible? Brazilian tech presence is real
    ('nigeria', 'healthcare'),# plausible
    ('venezuela', 'tech'),    # plausible
    ('canada', 'tech'),       # plausible
    ('ireland', 'h-1b'),      # less likely; few Irish on H-1B
    ('germany', 'agriculture'), # unlikely
]

for s, t in candidate_edges:
    try:
        score = link_score(s, t, kv=kv)
        print(f'  {score:.3f}  {s} → {t}')
    except KeyError as e:
        print(f'  skip ({e})')

## 4. 2D visualization with PCA

In [ ]:
from sklearn.decomposition import PCA

from migration_atlas.graph.build import build_default
G = build_default().graph

ids = [n for n in kv.key_to_index]
vecs = np.array([kv[n] for n in ids])
kinds = [G.nodes[n]['kind'] for n in ids]

pca = PCA(n_components=2, random_state=42)
xy = pca.fit_transform(vecs)

color_by_kind = {
    'country': '#c2410c', 'visa': '#1e3a8a',
    'law': '#581c87', 'industry': '#115e59',
}

fig, ax = plt.subplots(figsize=(10, 7))
for kind in set(kinds):
    mask = [k == kind for k in kinds]
    ax.scatter(xy[mask, 0], xy[mask, 1],
               c=color_by_kind.get(kind, '#999'), s=70,
               edgecolor='#0c0a09', linewidth=1, label=kind, alpha=0.85)
for i, nid in enumerate(ids):
    ax.annotate(G.nodes[nid].get('name', nid), (xy[i, 0], xy[i, 1]),
                fontsize=7, alpha=0.7, xytext=(4, 4), textcoords='offset points')
ax.legend(loc='best')
ax.set_title('Node2Vec embeddings — PCA projection')
ax.set_xlabel('PC 1')
ax.set_ylabel('PC 2')
plt.tight_layout()
plt.show()

## 5. Where this goes next

- **Use embeddings as features** in downstream ML — e.g., concatenate the country embedding with macro covariates as input to the LSTM forecaster.
- **Cluster countries** with k-means or HDBSCAN to discover "migration archetypes" (refugee-driven vs. labor-driven vs. family-driven).
- **Try p ≠ q** to bias toward structural roles (BFS-like, p < 1, q > 1) vs. communities (DFS-like, p > 1, q < 1).